In [1]:
!pip install pandas numpy scikit-learn tensorflow joblib -q
print("install completed")

install completed


In [2]:
import pandas as pd
import numpy as np
import random
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import joblib
import os

np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

def calculate_bmr(age, height_cm, weight_kg, metabolic_profile):

    # Harris-Benedict Formula for calculating BMR
    # metabolicProfile: PROFILE_1=Male Metabolism, PROFILE_2=Female Metabolism

    is_male = (metabolic_profile == 'PROFILE_1')

    if is_male:
        return 88.362 + (13.397 * weight_kg) + (4.799 * height_cm) - (5.677 * age)
    else:
        return 447.593 + (9.247 * weight_kg) + (3.098 * height_cm) - (4.330 * age)


def calculate_tdee(bmr, activity_level):
    multipliers = {
        'SEDENTARY': 1.2, 'LIGHT': 1.375, 'MODERATE': 1.55,
        'ACTIVE': 1.725, 'VERY_ACTIVE': 1.9
    }
    return bmr * multipliers[activity_level]


def calculate_target_calories(tdee, goal):
    if goal == 'LOSE':
        return tdee * 0.85
    elif goal == 'GAIN':
        return tdee * 1.15
    else:
        return tdee


def calculate_macros(calories, goal, diet_pref, weight_kg):
    # protein
    if goal == 'LOSE':
        protein_per_kg = 2.2
    elif goal == 'GAIN':
        protein_per_kg = 2.0
    else:
        protein_per_kg = 1.6

    if diet_pref == 'HIGH_PROTEIN':
        protein_per_kg *= 1.15
    elif diet_pref == 'VEGETARIAN':
        protein_per_kg *= 0.9

    protein_g = weight_kg * protein_per_kg
    fat_g = (calories * 0.25) / 9
    remaining_cals = calories - (protein_g * 4) - (fat_g * 9)
    carbs_g = max(0, remaining_cals / 4)

    protein_g *= random.uniform(0.95, 1.05)
    carbs_g *= random.uniform(0.95, 1.05)
    fat_g *= random.uniform(0.95, 1.05)

    return protein_g, carbs_g, fat_g


def calculate_workout_params(age, weight_kg, goal, activity_level, metabolic_profile):
    if age < 25:
        base_intensity = 0.5
    elif age < 40:
        base_intensity = 0.6
    else:
        base_intensity = 0.4

    if goal == 'GAIN':
        base_intensity += 0.15
    elif goal == 'LOSE':
        base_intensity += 0.05

    activity_bonus = {
        'SEDENTARY': -0.1, 'LIGHT': 0.0, 'MODERATE': 0.05,
        'ACTIVE': 0.1, 'VERY_ACTIVE': 0.15
    }
    base_intensity += activity_bonus[activity_level]

    intensity = base_intensity * random.uniform(0.9, 1.1)
    intensity = np.clip(intensity, 0.0, 1.0)

    if age < 30:
        split_choice = random.choices([1, 2], weights=[0.4, 0.6])[0]
    elif age < 50:
        split_choice = random.choices([0, 1, 2], weights=[0.2, 0.6, 0.2])[0]
    else:
        split_choice = random.choices([0, 1], weights=[0.8, 0.2])[0]

    return intensity, split_choice

print("complete")

complete


In [3]:
def generate_training_data(num_samples=10000):
    data = []

    goals = ['LOSE', 'MAINTAIN', 'GAIN']
    activity_levels = ['SEDENTARY', 'LIGHT', 'MODERATE', 'ACTIVE', 'VERY_ACTIVE']
    diet_prefs = ['BALANCED', 'HIGH_PROTEIN', 'VEGETARIAN', 'NO_PREFERENCE']
    metabolic_profiles = ['PROFILE_1', 'PROFILE_2']

    normal_samples = int(num_samples * 0.9)

    for _ in range(normal_samples):
        age = random.randint(18, 60)
        height_cm = random.uniform(150, 200)
        weight_kg = random.uniform(45, 120)
        goal = random.choice(goals)
        activity = random.choice(activity_levels)
        diet_pref = random.choice(diet_prefs)
        metabolic_profile = random.choice(metabolic_profiles)

        if random.random() < 0.05:
            weight_kg *= random.uniform(0.92, 1.08)
            height_cm *= random.uniform(0.98, 1.02)

        bmr = calculate_bmr(age, height_cm, weight_kg, metabolic_profile)
        tdee = calculate_tdee(bmr, activity)
        target_calories = calculate_target_calories(tdee, goal)

        metabolic_variation = random.uniform(0.88, 1.12)
        target_calories *= metabolic_variation

        protein_g, carbs_g, fat_g = calculate_macros(
            target_calories, goal, diet_pref, weight_kg
        )

        if diet_pref == 'HIGH_PROTEIN' and random.random() < 0.4:
            protein_g *= random.uniform(0.85, 0.95)
            carbs_g *= random.uniform(1.05, 1.15)
        elif diet_pref == 'VEGETARIAN' and random.random() < 0.3:
            protein_g *= random.uniform(0.9, 1.0)

        workout_intensity, workout_type = calculate_workout_params(
            age, weight_kg, goal, activity, metabolic_profile
        )

        intensity_variation = random.uniform(0.92, 1.08)
        workout_intensity *= intensity_variation
        workout_intensity = np.clip(workout_intensity, 0.0, 1.0)

        data.append({
            'age': int(age),
            'heightCm': round(height_cm, 2),
            'weightKg': round(weight_kg, 2),
            'activityLevel': activity,
            'goal': goal,
            'dietPref': diet_pref,
            'metabolicProfile': metabolic_profile,
            'caloriesKcal': int(target_calories),
            'proteinG': int(protein_g),
            'carbsG': int(carbs_g),
            'fatG': int(fat_g),
            'workoutIntensity': round(workout_intensity, 4),
            'workoutType': int(workout_type)
        })

    extreme_samples = num_samples - normal_samples

    for _ in range(extreme_samples):
        scenario = random.choice(['underweight', 'obese', 'elderly', 'athlete', 'beginner'])

        if scenario == 'underweight':
            age = random.randint(18, 35)
            height_cm = random.uniform(160, 185)
            weight_kg = random.uniform(42, 55)
            goal = 'GAIN'
            activity = random.choice(['SEDENTARY', 'LIGHT'])
            diet_pref = 'HIGH_PROTEIN'
            metabolic_profile = random.choice(metabolic_profiles)

        elif scenario == 'obese':
            age = random.randint(30, 55)
            height_cm = random.uniform(155, 180)
            weight_kg = random.uniform(90, 118)
            goal = 'LOSE'
            activity = random.choice(['SEDENTARY', 'LIGHT'])
            diet_pref = random.choice(['BALANCED', 'HIGH_PROTEIN'])
            metabolic_profile = random.choice(metabolic_profiles)

        elif scenario == 'elderly':
            age = random.randint(55, 60)
            height_cm = random.uniform(155, 175)
            weight_kg = random.uniform(55, 85)
            goal = random.choice(['MAINTAIN', 'LOSE'])
            activity = random.choice(['SEDENTARY', 'LIGHT'])
            diet_pref = random.choice(diet_prefs)
            metabolic_profile = random.choice(metabolic_profiles)

        elif scenario == 'athlete':
            age = random.randint(20, 35)
            height_cm = random.uniform(165, 195)
            weight_kg = random.uniform(65, 95)
            goal = random.choice(['GAIN', 'MAINTAIN'])
            activity = random.choice(['ACTIVE', 'VERY_ACTIVE'])
            diet_pref = 'HIGH_PROTEIN'
            metabolic_profile = random.choice(metabolic_profiles)

        else: #beginner
            age = random.randint(18, 40)
            height_cm = random.uniform(155, 185)
            weight_kg = random.uniform(50, 95)
            goal = random.choice(goals)
            activity = 'SEDENTARY'
            diet_pref = 'NO_PREFERENCE'
            metabolic_profile = random.choice(metabolic_profiles)

        bmr = calculate_bmr(age, height_cm, weight_kg, metabolic_profile)
        tdee = calculate_tdee(bmr, activity)
        target_calories = calculate_target_calories(tdee, goal)

        metabolic_variation = random.uniform(0.85, 1.15)
        target_calories *= metabolic_variation

        protein_g, carbs_g, fat_g = calculate_macros(
            target_calories, goal, diet_pref, weight_kg
        )

        if scenario == 'athlete':
            protein_g *= random.uniform(1.1, 1.2)
            carbs_g *= random.uniform(0.95, 1.0)

        workout_intensity, workout_type = calculate_workout_params(
            age, weight_kg, goal, activity, metabolic_profile
        )

        if scenario == 'elderly':
            workout_intensity *= random.uniform(0.7, 0.85)
            workout_type = 0
        elif scenario == 'athlete':
            workout_intensity *= random.uniform(1.05, 1.15)

        workout_intensity = np.clip(workout_intensity, 0.0, 1.0)

        data.append({
            'age': int(age),
            'heightCm': round(height_cm, 2),
            'weightKg': round(weight_kg, 2),
            'activityLevel': activity,
            'goal': goal,
            'dietPref': diet_pref,
            'metabolicProfile': metabolic_profile,
            'caloriesKcal': int(target_calories),
            'proteinG': int(protein_g),
            'carbsG': int(carbs_g),
            'fatG': int(fat_g),
            'workoutIntensity': round(workout_intensity, 4),
            'workoutType': int(workout_type)
        })

    return pd.DataFrame(data)


df = generate_training_data(10000)

print("\nThe first five lines of data:")
print(df.head())

print("\nStatistical data:")
print(df[['caloriesKcal', 'proteinG', 'carbsG', 'fatG', 'workoutIntensity']].describe())

print("\nTarget distribution:")
print(df['goal'].value_counts())

print("\nActivity Level Distribution:")
print(df['activityLevel'].value_counts())


The first five lines of data:
   age  heightCm  weightKg activityLevel      goal       dietPref  \
0   58    155.57    100.62         LIGHT      LOSE   HIGH_PROTEIN   
1   32    172.46     65.86         LIGHT      LOSE  NO_PREFERENCE   
2   20    186.49     85.22     SEDENTARY  MAINTAIN     VEGETARIAN   
3   41    158.13     71.65      MODERATE      GAIN       BALANCED   
4   21    161.45     47.41        ACTIVE  MAINTAIN     VEGETARIAN   

  metabolicProfile  caloriesKcal  proteinG  carbsG  fatG  workoutIntensity  \
0        PROFILE_1          2370       219     198    62            0.4259   
1        PROFILE_2          1581       148     145    43            0.6079   
2        PROFILE_2          2173       108     277    59            0.3761   
3        PROFILE_1          2634       142     342    76            0.6051   
4        PROFILE_1          2631        70     417    74            0.6253   

   workoutType  
0            0  
1            1  
2            1  
3            1  


In [4]:
le_activity = LabelEncoder()
le_goal = LabelEncoder()
le_diet = LabelEncoder()
le_metabolic = LabelEncoder()

df['activityLevel_encoded'] = le_activity.fit_transform(df['activityLevel'])
df['goal_encoded'] = le_goal.fit_transform(df['goal'])
df['dietPref_encoded'] = le_diet.fit_transform(df['dietPref'])
df['metabolicProfile_encoded'] = le_metabolic.fit_transform(df['metabolicProfile'])

feature_columns = [
    'age', 'heightCm', 'weightKg',
    'activityLevel_encoded', 'goal_encoded',
    'dietPref_encoded', 'metabolicProfile_encoded'
]

X = df[feature_columns].values

y_workout = df[['workoutIntensity', 'workoutType']].values
y_meal = df[['caloriesKcal', 'proteinG', 'carbsG', 'fatG']].values

# Standardisation
print("Standardising data")
scaler_X = StandardScaler()
scaler_y_workout = StandardScaler()
scaler_y_meal = StandardScaler()

X_scaled = scaler_X.fit_transform(X)
y_workout_scaled = scaler_y_workout.fit_transform(y_workout)
y_meal_scaled = scaler_y_meal.fit_transform(y_meal)

X_train, X_test, y_workout_train, y_workout_test, y_meal_train, y_meal_test = train_test_split(
    X_scaled, y_workout_scaled, y_meal_scaled,
    test_size=0.2, random_state=42
)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"Target number of workout: {y_workout_train.shape[1]}")
print(f"Target number of meal: {y_meal_train.shape[1]}")

Standardising data
Training set size: (8000, 7)
Test set size: (2000, 7)
Target number of workout: 2
Target number of meal: 4


In [6]:
# meal model
#input(7): age, height, weight, activity level, fitness goal, dietary preference
#output(4): calories, protein, carbs, fat

def create_meal_model_ablation(input_dim):

    user_input = keras.Input(shape=(input_dim,), name='user_input')


    x = layers.Dense(128, activation='relu', name='meal_dense_1')(user_input)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(64, activation='relu', name='meal_dense_2')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Dense(32, activation='relu', name='meal_features')(x)

    # output
    outputs = layers.Dense(4, activation='linear', name='nutrition_output')(x)

    model = keras.Model(
        inputs=user_input,
        outputs=outputs
    )

    model.compile(
        optimizer='adam',
        loss='mse',
        metrics=['mae']
    )

    return model


meal_model = create_meal_model_ablation(X_train.shape[1])
print("\n meal model architecture (Ablation):")
meal_model.summary()

early_stop_meal = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

#training meal model
meal_history = meal_model.fit(
    X_train,
    y_meal_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop_meal],
    verbose=1
)

# Evaluate
meal_loss, meal_mae = meal_model.evaluate(
    X_test,
    y_meal_test
)
print(f"\n Meal model training completed(Ablation)")
print(f" test set MAE: {meal_mae:.4f}")


 meal model architecture (Ablation):


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ user_input (InputLayer)         │ (None, 7)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ meal_dense_1 (Dense)            │ (None, 128)            │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ meal_dense_2 (Dense)            │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ meal_features (Dense)           │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ nutrition_output (Dense)        │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,260 (47.89 KB)

 Trainable params: 11,876 (46.39 KB)

 Non-trainable params: 384 (1.50 KB)

Epoch 1/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - loss: 0.8277 - mae: 0.7041 - val_loss: 0.5678 - val_mae: 0.5857
Epoch 2/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.4632 - mae: 0.5354 - val_loss: 0.2930 - val_mae: 0.4162
Epoch 3/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.3609 - mae: 0.4734 - val_loss: 0.2120 - val_mae: 0.3612
Epoch 4/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.3139 - mae: 0.4392 - val_loss: 0.1743 - val_mae: 0.3257
Epoch 5/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.2869 - mae: 0.4190 - val_loss: 0.1767 - val_mae: 0.3250
Epoch 6/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.2588 - mae: 0.3990 - val_loss: 0.1565 - val_mae: 0.3082
Epoch 7/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.2439 - mae: 0.3886 - val_loss: 0.1472 - val_mae: 0.2984
Epoch 8/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.2295 - mae: 0.3771 - val_loss: 0.1432 - val_mae: 0.2939
Epoch 9/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/